# Lesson 15 Lab — TorchAO INT4 Weight-Only Quantization

**Puzzle:** Can a PyTorch-native INT4 conversion reduce storage and still lose on latency?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

TorchAO conversion replaces or wraps eligible `Linear` weights with a packed tensor subclass/configuration. The Python module, packed storage, and selected matmul kernel are three inspectable layers.

### Core mechanism

INT4 weight-only compute conceptually reads packed codes and group scales while BF16 activations enter the linear operation. Modern TorchAO versions may choose among packing formats and external kernel libraries such as MSLK.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "15-torchao-int4"
device = require_cuda()
torch.manual_seed(2026 + 15)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Packing reduces persistent bytes, but conversion dependencies, scale handling, small-batch overhead, and unsupported shapes can erase latency gains. Version compatibility is part of the result.

### What this code tests

The notebook attempts the documented native configuration inside an explicit compatibility boundary and records the exact failure class when the path cannot execute.

**Experiment:** Convert a BF16 linear layer with TorchAO INT4, record the resulting module type, compare output error, and time both paths.

**Declared evidence label:** `compatibility-probe`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import copy, importlib.util
available=importlib.util.find_spec("torchao") is not None; details={"torchao_installed":available}
if available:
    try:
        from torchao.quantization import Int4WeightOnlyConfig, quantize_
        layer=torch.nn.Linear(4096,4096,bias=False,device=device,dtype=torch.bfloat16); candidate=copy.deepcopy(layer)
        x=torch.randn(8,4096,device=device,dtype=torch.bfloat16); ref=layer(x)
        quantize_(candidate,Int4WeightOnlyConfig(group_size=128)); out=candidate(x)
        details.update({"conversion":"success","module_type":type(candidate).__name__,"output_error":error_metrics(ref,out),
                        "bf16_timing":cuda_benchmark(lambda:layer(x),warmup=5,repeats=20),
                        "int4_timing":cuda_benchmark(lambda:candidate(x),warmup=5,repeats=20)})
    except Exception as exc:
        details.update({"conversion":"failed","error_type":type(exc).__name__,"error_message":str(exc)[:240]})
result=base_result(15,"native-backend" if details.get("conversion")=="success" else "compatibility-probe")
outcome=("TorchAO INT4 converted and executed; output error and latency were measured."
         if details.get("conversion")=="success" else
         "TorchAO was installed, but the native INT4 path did not execute; the dependency failure is preserved as a compatibility result.")
result.update({"torchao":details,"conclusion":outcome})


W0807 22:45:46.374000 451982 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 3. Inspect the evidence

Require conversion success, storage accounting, output error, and repeated latency. A missing TorchAO install becomes an explicit compatibility result.

### Acceptance and rollback gate

Require successful import/conversion, quantized tensor/module identity, storage accounting, operator evidence, output error, and repeated latency. Preserve dependency failure rather than falling back silently.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "TorchAO was installed, but the native INT4 path did not execute; the dependency failure is preserved as a compatibility result.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:46+00:00",
  "lesson": 15,
  "schema_version": 1,
  "torchao": {
    "conversion": "failed",
    "error_message": "Requires mslk >= 1.0.0",
    "error_type": "ImportError",
    "torchao_installed": true
  }
}
Saved: artifacts/rtx5090-result.json


## 4. Explain the result

Treat TorchAO INT4 as a measured backend path, not a universal performance property of four-bit weights.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).